# Train DataLoader 검증 노트북 (Colab 완결판)

`new_dataset.py` / `new_datamodule.py` 검증용. **셀 순서대로** 실행하세요.

| Step | 내용 | 필수 |
|------|------|------|
| **Colab** | Drive 마운트 + `git clone`/`git pull` + pip | Colab만 |
| 0 | 환경 설정 (`data/` → Drive) | ✓ |
| helpers | 다운로드/압축 유틸 | ✓ |
| 1 | Zenodo dev set → `data/dev_set/` | dataloader 스모크 시 |
| 2 | SpAudSyn | dataloader 스moke 시 |
| 3–4 | Semantic Hearing 다운로드 + dev_set 추가 | 선택 |
| 5 | 경로 검증 | dataloader 스moke 시 |
| **unittest** | **`test_new_*.py` 전용 셀** | ✓ (데이터 불필요) |
| 7–8 | train/val dataloader 스mo크 | Step 1–2·5 후 |

> **unittest만**: Colab → 0 → helpers → **unittest** 셀 (Step 1–5 skip 가능)
>
> **전체 검증**: Colab → 0 → helpers → 1 → 2 → (3–4) → 5 → unittest → 7 → 8

## [Colab] Drive 마운트 + repo clone / pull

로컬 PC면 **skip**. Repo 최신 반영: `git pull`

In [ ]:
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = "https://github.com/Mockdd/dcase2026_task4.git"
COLAB_REPO_DIR = Path("/content/dcase2026_task4")
DRIVE_DATA_DIR = Path("/content/drive/MyDrive/dcase2026_task4/data")

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE_DATA_DIR.mkdir(parents=True, exist_ok=True)

    if (COLAB_REPO_DIR / ".git").exists():
        print("[git pull]")
        subprocess.run(["git", "pull"], cwd=COLAB_REPO_DIR, check=True)
    elif not (COLAB_REPO_DIR / "src" / "datamodules" / "new_dataset.py").exists():
        print("[git clone]", REPO_URL)
        subprocess.run(["git", "clone", REPO_URL, str(COLAB_REPO_DIR)], check=True)

    %cd {COLAB_REPO_DIR}
    !pip install -q lightning pyyaml soundfile librosa scipy python-sofa tqdm
    !git log -1 --oneline
    print("\nColab OK | code:", COLAB_REPO_DIR)
    print("         | data:", DRIVE_DATA_DIR)
else:
    print("Not Colab — skip this cell.")

## Step 0. 환경 설정

In [ ]:
import os
import re
import shutil
import subprocess
import sys
import tarfile
import unittest
import urllib.request
import zipfile
from io import StringIO
from pathlib import Path

import torch
import yaml

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


def _data_is_scaffold_only(data_path: Path) -> bool:
    if not data_path.exists():
        return True
    return not any(p.is_file() and p.stat().st_size > 0 for p in data_path.rglob("*"))


if IN_COLAB:
    PROJECT_ROOT = Path("/content/dcase2026_task4")
    DATA_DIR = Path("/content/drive/MyDrive/dcase2026_task4/data")
else:
    PROJECT_ROOT = Path.cwd()
    if not (PROJECT_ROOT / "src" / "datamodules" / "new_dataset.py").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent
    DATA_DIR = PROJECT_ROOT / "data"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

LOCAL_DATA = PROJECT_ROOT / "data"
if IN_COLAB:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    if LOCAL_DATA.is_symlink() and LOCAL_DATA.resolve() == DATA_DIR.resolve():
        print("[ok] symlink already set")
    elif _data_is_scaffold_only(LOCAL_DATA):
        if LOCAL_DATA.exists() or LOCAL_DATA.is_symlink():
            LOCAL_DATA.unlink() if LOCAL_DATA.is_symlink() else shutil.rmtree(LOCAL_DATA)
        LOCAL_DATA.symlink_to(DATA_DIR, target_is_directory=True)
        print("[ok] symlink created")
    else:
        shutil.copytree(LOCAL_DATA, DATA_DIR, dirs_exist_ok=True)
        shutil.rmtree(LOCAL_DATA)
        LOCAL_DATA.symlink_to(DATA_DIR, target_is_directory=True)
        print("[ok] local data moved to Drive + symlink")
    print("symlink:", LOCAL_DATA, "->", DATA_DIR.resolve())

DEV_SET_DIR = DATA_DIR / "dev_set"
DOWNLOAD_DIR = DATA_DIR / "downloads"
ZENODO_DIR = DOWNLOAD_DIR / "zenodo"
SEMHEAR_TAR = DATA_DIR / "BinauralCuratedDataset.tar"
SEMHEAR_DIR = DATA_DIR / "BinauralCuratedDataset"
SPAUDSYN_DIR = PROJECT_ROOT / "src" / "modules" / "spatial_audio_synthesizer"

for d in (DATA_DIR, DOWNLOAD_DIR, ZENODO_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT.resolve())
print("DATA_DIR    :", DATA_DIR.resolve())
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

## helpers. 다운로드 / Zenodo split-zip 유틸

In [ ]:
def describe_batch(batch, title="batch"):
    print(f"\n=== {title} ===")
    for k, v in batch.items():
        if isinstance(v, torch.Tensor):
            print(f"  {k:20s} shape={tuple(v.shape)} dtype={v.dtype}")
        elif isinstance(v, list):
            print(f"  {k:20s} list[len={len(v)}]")
        else:
            print(f"  {k:20s} {type(v).__name__}")


def download_url(url: str, dest: Path, label: str = "") -> None:
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > 0:
        print(f"[skip] {label or dest.name}")
        return
    print(f"[download] {label or dest.name}")
    def _progress(n, size, total):
        if total > 0:
            print(f"\r  {min(n*size,total)/total*100:5.1f}%", end="", flush=True)
    urllib.request.urlretrieve(url, dest, reporthook=_progress)
    print("\n  done.")


def _ensure_archive_tools():
    pkgs = [("unzip", "unzip"), ("zip", "zip"), ("p7zip-full", "7z")]
    need = [pkg for pkg, cmd in pkgs if not shutil.which(cmd)]
    if need:
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "-qq", *need], check=True)


def _unzip_test(path: Path) -> bool:
    return subprocess.run(["unzip", "-t", str(path)], capture_output=True).returncode == 0


def _extract_with_unzip(zip_path: Path, dest_dir: Path) -> None:
    dest_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run(["unzip", "-o", "-q", str(zip_path), "-d", str(dest_dir)], check=True)


def _extract_with_7z(zip_path: Path, dest_dir: Path) -> None:
    dest_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run(["7z", "x", "-y", f"-o{dest_dir}", zip_path.name], cwd=zip_path.parent, check=True)


def prepare_zenodo_dev_set(zenodo_dir: Path, extract_root: Path) -> None:
    """Zenodo split zip → dev_set. zip -s 0 / unzip / 7z fallback. Python zipfile 사용 안 함."""
    _ensure_archive_tools()
    split_zip = zenodo_dir / "DCASE2026Task4Dataset.zip"
    merged_zip = zenodo_dir / "DCASE2026Task4DatasetFull.zip"
    if not split_zip.exists():
        raise FileNotFoundError(f"Missing {split_zip}")

    if merged_zip.exists() and not _unzip_test(merged_zip):
        print(f"[cleanup] bad merged zip: {merged_zip.name}")
        merged_zip.unlink()

    marker = extract_root / ".extracted"
    if marker.exists() and not any(extract_root.rglob("**/valid.json")):
        marker.unlink()
    if extract_root.exists() and not any(extract_root.rglob("**/valid.json")):
        shutil.rmtree(extract_root)

    if (DEV_SET_DIR / "metadata" / "valid.json").exists():
        print("[skip] dev_set already installed")
        return

    if _extract_done(extract_root):
        print("[skip] extract folder ready")
    else:
        extract_root.mkdir(parents=True, exist_ok=True)
        ok = False
        print("[extract A] unzip split archive")
        try:
            _extract_with_unzip(split_zip, extract_root)
            ok = _extract_done(extract_root)
        except subprocess.CalledProcessError as e:
            print("  failed:", e)
        if not ok:
            if not merged_zip.exists():
                print("[merge] zip -s 0")
                subprocess.run(
                    ["zip", "-s", "0", split_zip.name, "--out", merged_zip.name],
                    cwd=zenodo_dir, check=True,
                )
            print("[extract B] unzip merged")
            _extract_with_unzip(merged_zip, extract_root)
            ok = _extract_done(extract_root)
        if not ok:
            print("[extract C] 7z")
            if extract_root.exists():
                shutil.rmtree(extract_root)
            extract_root.mkdir(parents=True)
            _extract_with_7z(split_zip, extract_root)
            ok = _extract_done(extract_root)
        if not ok:
            raise RuntimeError("Zenodo extract failed — check .z01-.z10 downloads")
        marker.write_text("ok", encoding="utf-8")

    install_dev_set_from_extract(extract_root)


def _extract_done(extract_root: Path) -> bool:
    return any(extract_root.rglob("**/valid.json")) if extract_root.exists() else False


def install_dev_set_from_extract(extract_root: Path) -> None:
    if (DEV_SET_DIR / "metadata" / "valid.json").exists():
        return
    candidates = [
        extract_root / "DCASE2026Task4Dataset" / "data" / "dev_set",
        extract_root / "data" / "dev_set",
    ]
    candidates += [p for p in extract_root.rglob("dev_set") if (p / "metadata" / "valid.json").exists()]
    src = next((p for p in candidates if p.exists()), None)
    if src is None:
        raise FileNotFoundError(f"dev_set not found under {extract_root}")
    print(f"[install] {src} -> {DEV_SET_DIR}")
    if DEV_SET_DIR.exists():
        shutil.rmtree(DEV_SET_DIR)
    shutil.copytree(src, DEV_SET_DIR)

print("helpers loaded.")

## Step 1. Zenodo DCASE dev set

[Zenodo 19328046](https://zenodo.org/records/19328046) — Drive `data/downloads/zenodo/`에 저장

In [ ]:
ZENODO_URLS = [
    line.strip()
    for line in (PROJECT_ROOT / "dev_set_zenodo.txt").read_text(encoding="utf-8").splitlines()
    if line.strip() and "lisence.pdf" not in line.strip().lower()
]
for url in ZENODO_URLS:
    fname = url.rsplit("/", 1)[-1]
    download_url(url, ZENODO_DIR / fname, label=fname)

extract_root = ZENODO_DIR / "extracted"
prepare_zenodo_dev_set(ZENODO_DIR, extract_root)
assert (DEV_SET_DIR / "metadata" / "valid.json").exists()
print("\nStep 1 OK:", DEV_SET_DIR)

## Step 2. SpAudSyn

In [ ]:
marker = SPAUDSYN_DIR / "spatial_audio_synthesizer.py"
if marker.exists():
    print(f"[skip] {SPAUDSYN_DIR}")
else:
    clone_dir = DOWNLOAD_DIR / "SpAudSyn_repo"
    if clone_dir.exists():
        shutil.rmtree(clone_dir)
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/nttcslab/SpAudSyn.git", str(clone_dir)], check=True)
    SPAUDSYN_DIR.parent.mkdir(parents=True, exist_ok=True)
    if SPAUDSYN_DIR.exists():
        shutil.rmtree(SPAUDSYN_DIR)
    shutil.copytree(clone_dir / "src", SPAUDSYN_DIR)
    print(f"[install] {SPAUDSYN_DIR}")
assert marker.exists()
print("\nStep 2 OK")

## Step 3. Semantic Hearing 다운로드

[SemanticHearing](https://github.com/vb000/SemanticHearing) — **unittest 불필요**, sound event 보강용

In [ ]:
SEMHEAR_URL = "https://semantichearing.cs.washington.edu/BinauralCuratedDataset.tar"
if SEMHEAR_DIR.exists() and any(SEMHEAR_DIR.iterdir()):
    print(f"[skip] {SEMHEAR_DIR}")
elif SEMHEAR_TAR.exists():
    with tarfile.open(SEMHEAR_TAR, "r") as tar:
        tar.extractall(DATA_DIR)
else:
    download_url(SEMHEAR_URL, SEMHEAR_TAR, label="BinauralCuratedDataset.tar")
    with tarfile.open(SEMHEAR_TAR, "r") as tar:
        tar.extractall(DATA_DIR)
if not SEMHEAR_DIR.exists():
    alt = next((p for p in DATA_DIR.glob("Binaural*") if p.is_dir()), None)
    if alt:
        globals()["SEMHEAR_DIR"] = alt
assert SEMHEAR_DIR.exists()
print("\nStep 3 OK:", SEMHEAR_DIR)

## Step 4. Semantic Hearing → dev_set 추가

In [ ]:
ADD_MARKER = DEV_SET_DIR / ".semhear_added"
COMMON = ["--target_sr=32000", "--amp_threshold=0.02", "--min_length=0.1", "--segment=10", "--shift=0.1"]

if ADD_MARKER.exists():
    print("[skip] Semantic Hearing already added")
else:
    bg = SEMHEAR_DIR / "bg_scaper_fmt"
    fsd = SEMHEAR_DIR / "FSD50K"
    assert bg.exists() and fsd.exists()
    subprocess.run([sys.executable, "add_interference.py", "--input_dir", str(bg),
        "--output_dir", str(DEV_SET_DIR / "interference"), *COMMON], cwd=PROJECT_ROOT, check=True)
    subprocess.run([sys.executable, "add_sound_event.py", "--input_dir", str(fsd),
        "--output_dir", str(DEV_SET_DIR / "sound_event"),
        "--pickup_json", str(DEV_SET_DIR / "config" / "FSD50K_config.json"), *COMMON],
        cwd=PROJECT_ROOT, check=True)
    ADD_MARKER.write_text("ok", encoding="utf-8")
print("\nStep 4 OK")

## Step 5. 경로 검증 (dataloader 스mo크용)

In [ ]:
REQUIRED = [
    "data/dev_set/sound_event/train", "data/dev_set/noise/train",
    "data/dev_set/interference/train", "data/dev_set/room_ir/train",
    "data/dev_set/metadata/valid.json",
    "src/modules/spatial_audio_synthesizer/spatial_audio_synthesizer.py",
]
missing = [r for r in REQUIRED if not (PROJECT_ROOT / r).exists()]
for r in REQUIRED:
    print(f"[{'OK' if r not in missing else 'MISSING'}] {r}")
if missing:
    print("\n[warn] missing paths — Step 7–8 skip 가능, unittest는 OK")
else:
    print("\nStep 5 OK")

---
## ★ unittest 전용 셀 (`test_new_dataset` + `test_new_datamodule`)

**dev set / Semantic Hearing / SpAudSyn 없이 실행 가능.**
Colab → Step 0 → helpers → **이 셀** 만으로도 검증 가능.

In [ ]:
def run_datamodule_tests(verbosity=2):
    loader = unittest.TestLoader()
    suite = unittest.TestSuite()
    suite.addTests(loader.discover(
        start_dir=str(PROJECT_ROOT / "tests" / "datamodules"),
        pattern="test_new_*.py",
        top_level_dir=str(PROJECT_ROOT),
    ))
    stream = StringIO()
    result = unittest.TextTestRunner(stream=stream, verbosity=verbosity).run(suite)
    print(stream.getvalue())
    print(f"\nRan {result.testsRun} | failures={len(result.failures)} | errors={len(result.errors)}")
    for label, items in [("FAILURES", result.failures), ("ERRORS", result.errors)]:
        if items:
            print(f"\n--- {label} ---")
            for test, tb in items:
                print(test, "\n", tb)
    return result

test_result = run_datamodule_tests()
assert not test_result.failures and not test_result.errors
print("\n★ unittest OK — new_dataset.py / new_datamodule.py (26 tests)")

## Step 7. Train DataModule 구성 (Step 1–2·5 완료 후)

In [ ]:
from src.datamodules.new_datamodule import DataModule

with open(PROJECT_ROOT / "config" / "label" / "m2dat_1c.yaml") as f:
    cfg = yaml.safe_load(f)
dm_args = cfg["datamodule"]["args"]
for split in ("train_dataloader", "val_dataloader"):
    if split in dm_args:
        ds = dm_args[split]["dataset"]
        ds["module"] = "src.datamodules.new_dataset"
        ds["main"] = "DatasetS3"

train_cfg = dm_args["train_dataloader"]
train_cfg.update(batch_size=2, num_workers=0, persistent_workers=False)
train_cfg["dataset"]["args"]["config"]["dataset_length"] = 8
val_cfg = dm_args.get("val_dataloader")
if val_cfg:
    val_cfg.update(batch_size=2, num_workers=0, persistent_workers=False)

dm = DataModule(train_dataloader=train_cfg, val_dataloader=val_cfg)
train_loader = dm.train_dataloader()
print("Step 7 OK | mode:", train_cfg["dataset"]["args"]["config"]["mode"])

## Step 8. Train / Val dataloader 스mo크 테스트

In [ ]:
batch = next(iter(train_loader))
describe_batch(batch, "train batch")
for key in ("mixture", "labels", "doas", "active"):
    assert key in batch and batch[key].dim() >= 2
if dm.val_dataset is not None:
    describe_batch(next(iter(dm.val_dataloader())), "val batch")
print("\nStep 8 OK — dataloader smoke test passed")